# FakeGuard AI - Fake News Detector
### NeuroLogic 26 Datathon

**Run cells ONE AT A TIME, top to bottom.**
- Cell 1 -> Cell 2 (restart) -> then Cell 3 onward
- GPU must be T4. Check: Settings -> Accelerator -> GPU T4 x1
- Dataset: Add your competition CSV via Notebook -> Add Data


## Cell 1 - Install / Fix Packages
Run once. Then run Cell 2 to restart kernel.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'peft'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.40.0', 'accelerate==0.29.3', 'datasets', 'scikit-learn', 'seaborn'])
print('Packages ready. Now run Cell 2 to restart kernel.')

## Cell 2 - Restart Kernel
Run once after Cell 1. After restart, **skip to Cell 3**.

In [ ]:
import IPython
print('Restarting kernel... after restart, begin from Cell 3.')
IPython.Application.instance().kernel.do_shutdown(restart=True)

## Cell 3 - GPU Check
Must show Tesla T4. If not, go to Settings -> Accelerator -> GPU T4 x1.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU found. Go to Settings > Accelerator > GPU T4 x1')
gpu_name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name}')
print(f'VRAM: {vram:.1f} GB')
if 'P100' in gpu_name:
    raise RuntimeError('Switch to T4. P100 is not supported.')
print('GPU ready.')

## Cell 4 - Load Competition Dataset
Loads the single competition CSV. Expects columns: title, text, subject, date, label.
We will DROP subject and date - they leak label information.

In [ ]:
import pandas as pd, glob, os

all_csvs = sorted(glob.glob('/kaggle/input/**/*.csv', recursive=True))
print('CSVs found in /kaggle/input/:')
for p in all_csvs:
    print(' ', p)

CSV_PATH = None
for p in all_csvs:
    try:
        peek = pd.read_csv(p, nrows=2)
        if 'label' in [c.lower().strip() for c in peek.columns]:
            CSV_PATH = p
            print(f'Auto-detected competition CSV: {p}')
            break
    except:
        pass

if CSV_PATH is None:
    raise FileNotFoundError('Could not auto-detect CSV. Set CSV_PATH manually above.')

df_raw = pd.read_csv(CSV_PATH, engine='python', on_bad_lines='skip')
df_raw.columns = [c.lower().strip() for c in df_raw.columns]
print(f'Loaded {len(df_raw):,} rows')
print(f'Columns: {df_raw.columns.tolist()}')
print(f'Label values: {df_raw["label"].unique()[:10]}')

## Cell 5 - Explore Data
Look at class balance, sample rows, and column types.

In [ ]:
import seaborn as sns, matplotlib.pyplot as plt

print('=== DATASET OVERVIEW ===')
print(f'Total rows   : {len(df_raw):,}')
print(f'Columns      : {df_raw.columns.tolist()}')
print()
print('=== LABEL DISTRIBUTION ===')
print(df_raw['label'].value_counts())
print()
print('=== SAMPLE ROWS ===')
print(df_raw[['title','text','label']].sample(3, random_state=42).to_string())
print()
print('=== NULL CHECK ===')
print(df_raw.isnull().sum())
print()

if 'subject' in df_raw.columns:
    print('=== WARNING: SUBJECT COLUMN LEAKS LABEL - WILL BE DROPPED ===')
    print(df_raw.groupby(['subject','label']).size().unstack(fill_value=0))

## Cell 6 - Clean, Deduplicate, Split

**What this cell does (explained simply):**
1. Drop subject and date (they cheat - they almost perfectly predict the label)
2. Use only title + text
3. Clean text (remove URLs, weird characters)
4. Remove duplicate articles
5. Convert labels to numbers (TRUE=1, FALSE=0)
6. Split into Train (85%) and Validation (15%) with zero leakage


In [ ]:
import re, hashlib
from sklearn.model_selection import train_test_split

df = df_raw[['title', 'text', 'label']].copy()
print(f'Step 1 - Kept only title, text, label: {len(df):,} rows')

df = df.dropna(subset=['title', 'text']).copy()
print(f'Step 2 - After dropping nulls: {len(df):,} rows')

def norm_label(v):
    v = str(v).strip().upper()
    if v in ('1', 'TRUE', 'REAL'): return 1
    if v in ('0', 'FALSE', 'FAKE'): return 0
    return None

df['label'] = df['label'].apply(norm_label)
df = df[df['label'].notna()].copy()
df['label'] = df['label'].astype(int)
print(f'Step 3 - Labels normalized: {df["label"].value_counts().to_dict()}')

def clean_text(t):
    if pd.isna(t): return ''
    t = str(t)
    t = t.encode('ascii', 'ignore').decode('ascii')
    t = re.sub(r'http\S+', ' ', t)
    t = re.sub(r"[^\w\s.,!?\-]", ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t

df['title_clean'] = df['title'].apply(clean_text)
df['text_clean']  = df['text'].apply(clean_text)
SEP = ' [SEP] '
df['combined'] = df['title_clean'] + SEP + df['text_clean']
df = df[df['combined'].str.len() > 30].copy()
print(f'Step 4 - Text cleaned: {len(df):,} rows')

before = len(df)
df['_key'] = df['combined'].str.lower().str.strip().apply(
    lambda x: hashlib.md5(x.encode('utf-8', errors='replace')).hexdigest()
)
df = df.drop_duplicates(subset='_key').drop(columns='_key').reset_index(drop=True)
print(f'Step 5 - Dedup: {before:,} -> {len(df):,} rows (removed {before-len(df):,} duplicates)')
print(f'       Label dist: {df["label"].value_counts().to_dict()}')

train_df, val_df = train_test_split(
    df[['combined', 'label']],
    test_size=0.15, random_state=42, stratify=df['label'], shuffle=True
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

make_key = lambda t: hashlib.md5(t.lower().strip().encode('utf-8', errors='replace')).hexdigest()
train_keys = set(train_df['combined'].apply(make_key))
val_keys   = set(val_df['combined'].apply(make_key))
leakage = len(train_keys & val_keys)
print(f'Step 6 - Split done. Leakage = {leakage} rows (must be 0)')
assert leakage == 0, f'Still leaking {leakage} rows!'

print(f'\nREADY')
print(f'   Train: {len(train_df):,} rows | {train_df["label"].value_counts().to_dict()}')
print(f'   Val  : {len(val_df):,} rows | {val_df["label"].value_counts().to_dict()}')
print(f'\nWARNING: Expected honest baseline: 85-92% (NOT 99-100%)')
print('   If baseline is 99%+, subject/date column may still be leaking.')

## Cell 7 - Honest Baseline (TF-IDF + Logistic Regression)

**What this does (simply):**
- Converts words to numbers (TF-IDF = how important each word is)
- Trains a simple Logistic Regression classifier
- Shows which words are most associated with FAKE vs REAL

**Expected result: 85-92% accuracy.** If you see 99%+, something is leaking.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

print('Training TF-IDF + Logistic Regression baseline...')

vectorizer = TfidfVectorizer(max_features=50000, ngram_range=(1, 2))
X_train = vectorizer.fit_transform(train_df['combined'])
X_val   = vectorizer.transform(val_df['combined'])

lr = LogisticRegression(max_iter=1000, random_state=42, C=1.0)
lr.fit(X_train, train_df['label'])
baseline_preds = lr.predict(X_val)
baseline_acc = accuracy_score(val_df['label'], baseline_preds)

print(f'\nBaseline TF-IDF + LR Accuracy: {baseline_acc:.4f}')
print()
print(classification_report(val_df['label'], baseline_preds, target_names=['FALSE (fake)', 'TRUE (real)']))

print('\n=== TOP 20 WORDS -> FAKE NEWS ===')
feature_names = np.array(vectorizer.get_feature_names_out())
coef = lr.coef_[0]
fake_idx = np.argsort(coef)[:20]
real_idx = np.argsort(coef)[-20:][::-1]
for w, c in zip(feature_names[fake_idx], coef[fake_idx]):
    print(f'  {w:<30} {c:.4f}')
print('\n=== TOP 20 WORDS -> REAL NEWS ===')
for w, c in zip(feature_names[real_idx], coef[real_idx]):
    print(f'  {w:<30} {c:.4f}')

BASELINE_ACC = baseline_acc
print(f'\nBaseline stored: {BASELINE_ACC:.4f}')
print('Now run Cell 8 to train RoBERTa (takes 30-40 min on T4)')

## Cell 8 - Train RoBERTa (Anti-Overfit Edition)

**What this does (simply):**
- Uses a pre-trained language model (RoBERTa) that already understands English
- Fine-tunes it on our fake news task
- Anti-overfitting settings prevent it from memorizing the training set

**Expected: ~94-97% honest validation accuracy. Takes ~30-40 min on T4.**

If it crashes: run Cell 8B to resume from checkpoint.

In [ ]:
import os, torch, numpy as np
from torch.utils.data import Dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoTokenizer, RobertaForSequenceClassification, RobertaConfig,
    TrainingArguments, Trainer, DataCollatorWithPadding, EarlyStoppingCallback
)

MODEL_SAVE = '/kaggle/working/roberta_fakenews'
os.makedirs(MODEL_SAVE, exist_ok=True)

class FakeNewsDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=512):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = self.tokenizer(
            str(row['combined']),
            truncation=True, max_length=self.max_length, padding=False
        )
        return {
            'input_ids': enc['input_ids'],
            'attention_mask': enc['attention_mask'],
            'labels': int(row['label'])
        }

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds, average='weighted')
    }

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained('roberta-base')

print('Loading RoBERTa with anti-overfitting dropout...')
config = RobertaConfig.from_pretrained(
    'roberta-base', num_labels=2,
    id2label={0: 'FALSE', 1: 'TRUE'},
    label2id={'FALSE': 0, 'TRUE': 1},
    hidden_dropout_prob=0.2,
    attention_probs_dropout_prob=0.2
)
model = RobertaForSequenceClassification.from_pretrained('roberta-base', config=config)
print(f'Model loaded. Parameters: {sum(p.numel() for p in model.parameters()):,}')

train_dataset = FakeNewsDataset(train_df, tokenizer)
val_dataset   = FakeNewsDataset(val_df, tokenizer)
print(f'Train: {len(train_dataset):,} samples | Val: {len(val_dataset):,} samples')

args = TrainingArguments(
    output_dir=MODEL_SAVE, num_train_epochs=5,
    per_device_train_batch_size=16, per_device_eval_batch_size=64,
    warmup_steps=1000, weight_decay=0.10, learning_rate=2e-5,
    max_grad_norm=1.0, eval_strategy='epoch', save_strategy='epoch',
    load_best_model_at_end=True, metric_for_best_model='accuracy',
    greater_is_better=True, fp16=torch.cuda.is_available(),
    report_to='none', logging_steps=100, seed=42, save_total_limit=2
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_dataset, eval_dataset=val_dataset,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print('=' * 60)
print('STARTING RoBERTa TRAINING')
print('  weight_decay=0.10 | warmup=1000 | dropout=0.2 | grad_clip=1.0')
print('  Early stopping patience=2 epochs')
print('  Expected time: 30-40 min on T4 GPU')
print('=' * 60)

train_result = trainer.train()
eval_metrics = trainer.evaluate()

trainer.save_model(MODEL_SAVE)
tokenizer.save_pretrained(MODEL_SAVE)

ROBERTA_ACC = eval_metrics['eval_accuracy']
ROBERTA_F1  = eval_metrics['eval_f1']

print(f'\nVal Accuracy : {ROBERTA_ACC:.4f}')
print(f'Val F1        : {ROBERTA_F1:.4f}')
try:
    print(f'Improvement over baseline: {ROBERTA_ACC - BASELINE_ACC:+.4f}')
except: pass
print(f'Model saved to {MODEL_SAVE}')
print('Now run Cell 9 to evaluate, then Cell 10 to generate submission.')

## Cell 8B - Resume Training (Only If Cell 8 Crashed)
Only run this if Cell 8 trained at least 1 epoch and then crashed.

In [ ]:
import os, torch, numpy as np
from torch.utils.data import Dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoTokenizer, RobertaForSequenceClassification, RobertaConfig,
    TrainingArguments, Trainer, DataCollatorWithPadding, EarlyStoppingCallback
)

MODEL_SAVE = '/kaggle/working/roberta_fakenews'
checkpoints = sorted(
    [d for d in os.listdir(MODEL_SAVE) if d.startswith('checkpoint-')],
    key=lambda x: int(x.split('-')[-1])
)
if not checkpoints:
    raise FileNotFoundError('No checkpoint found. Run Cell 8 for a fresh start.')
latest_ckpt = os.path.join(MODEL_SAVE, checkpoints[-1])
print(f'Resuming from: {latest_ckpt}')

class FakeNewsDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=512):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = self.tokenizer(str(row['combined']), truncation=True, max_length=self.max_length, padding=False)
        return {'input_ids': enc['input_ids'], 'attention_mask': enc['attention_mask'], 'labels': int(row['label'])}

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, preds), 'f1': f1_score(labels, preds, average='weighted')}

tokenizer = AutoTokenizer.from_pretrained('roberta-base')
config = RobertaConfig.from_pretrained(latest_ckpt, num_labels=2,
    id2label={0:'FALSE',1:'TRUE'}, label2id={'FALSE':0,'TRUE':1},
    hidden_dropout_prob=0.2, attention_probs_dropout_prob=0.2)
model = RobertaForSequenceClassification.from_pretrained(latest_ckpt, config=config)

train_dataset = FakeNewsDataset(train_df, tokenizer)
val_dataset   = FakeNewsDataset(val_df, tokenizer)

args = TrainingArguments(
    output_dir=MODEL_SAVE, num_train_epochs=5, per_device_train_batch_size=16,
    per_device_eval_batch_size=64, warmup_steps=1000, weight_decay=0.10,
    learning_rate=2e-5, max_grad_norm=1.0, eval_strategy='epoch',
    save_strategy='epoch', load_best_model_at_end=True,
    metric_for_best_model='accuracy', greater_is_better=True,
    fp16=torch.cuda.is_available(), report_to='none', logging_steps=100, seed=42, save_total_limit=2
)
trainer = Trainer(model=model, args=args, train_dataset=train_dataset, eval_dataset=val_dataset,
    data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])

train_result = trainer.train(resume_from_checkpoint=latest_ckpt)
eval_metrics = trainer.evaluate()
trainer.save_model(MODEL_SAVE)
tokenizer.save_pretrained(MODEL_SAVE)
print(f'Resume complete. Val Accuracy: {eval_metrics["eval_accuracy"]:.4f}')

## Cell 9 - Evaluate Model
Shows confusion matrix, accuracy, and F1 score.

In [ ]:
import numpy as np, torch, matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, RobertaForSequenceClassification, DataCollatorWithPadding

MODEL_SAVE = '/kaggle/working/roberta_fakenews'

class FakeNewsDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=512):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = self.tokenizer(str(row['combined']), truncation=True, max_length=self.max_length, padding=False)
        return {'input_ids': enc['input_ids'], 'attention_mask': enc['attention_mask'], 'labels': int(row['label'])}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained(MODEL_SAVE)
model = RobertaForSequenceClassification.from_pretrained(MODEL_SAVE).to(device)
model.eval()

val_dataset = FakeNewsDataset(val_df, tokenizer)
collator = DataCollatorWithPadding(tokenizer)
loader = DataLoader(val_dataset, batch_size=64, collate_fn=collator)

all_preds, all_labels = [], []
with torch.no_grad():
    for batch in loader:
        outputs = model(input_ids=batch['input_ids'].to(device), attention_mask=batch['attention_mask'].to(device))
        all_preds.extend(torch.argmax(outputs.logits, dim=-1).cpu().numpy())
        all_labels.extend(batch['labels'].numpy())

acc = accuracy_score(all_labels, all_preds)
print(f'RoBERTa Val Accuracy: {acc:.4f}')
try:
    print(f'   Baseline Accuracy  : {BASELINE_ACC:.4f}')
    print(f'   Improvement        : {acc - BASELINE_ACC:+.4f}')
except: pass
print()
print(classification_report(all_labels, all_preds, target_names=['FALSE (fake)', 'TRUE (real)']))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred FAKE', 'Pred REAL'], yticklabels=['True FAKE', 'True REAL'])
plt.title(f'Confusion Matrix | Accuracy: {acc:.4f}')
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix.png', dpi=150)
plt.show()
print('Confusion matrix saved to /kaggle/working/confusion_matrix.png')

## Cell 10 - Generate submission.csv

Predicts on the unlabeled test rows and saves submission.csv.
Upload /kaggle/working/submission.csv to the competition leaderboard.

In [ ]:
import pandas as pd, torch, numpy as np, re, glob
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, RobertaForSequenceClassification, DataCollatorWithPadding

MODEL_SAVE = '/kaggle/working/roberta_fakenews'

all_csvs = sorted(glob.glob('/kaggle/input/**/*.csv', recursive=True))
test_path = None
for p in all_csvs:
    try:
        peek = pd.read_csv(p, nrows=2)
        peek.columns = [c.lower().strip() for c in peek.columns]
        if 'label' not in peek.columns and 'text' in peek.columns:
            test_path = p
            print(f'Auto-detected test CSV: {p}')
            break
    except: pass

if test_path is None:
    print('WARNING: No unlabeled test file found. Skipping submission generation.')
else:
    df_test = pd.read_csv(test_path, engine='python', on_bad_lines='skip')
    df_test.columns = [c.lower().strip() for c in df_test.columns]

    def clean_text(t):
        if pd.isna(t): return ''
        t = str(t).encode('ascii', 'ignore').decode('ascii')
        t = re.sub(r'http\S+', ' ', t)
        t = re.sub(r"[^\w\s.,!?\-]", ' ', t)
        return re.sub(r'\s+', ' ', t).strip()

    title_col = 'title' if 'title' in df_test.columns else None
    text_col  = 'text'  if 'text'  in df_test.columns else None
    df_test['combined'] = (
        (df_test[title_col].apply(clean_text) + ' [SEP] ' if title_col else '') +
        (df_test[text_col].apply(clean_text) if text_col else '')
    )

    class TestDataset(Dataset):
        def __init__(self, texts, tokenizer, max_length=512):
            self.texts = texts
            self.tokenizer = tokenizer
            self.max_length = max_length
        def __len__(self): return len(self.texts)
        def __getitem__(self, idx):
            enc = self.tokenizer(str(self.texts[idx]), truncation=True, max_length=self.max_length, padding=False)
            return {'input_ids': enc['input_ids'], 'attention_mask': enc['attention_mask']}

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tokenizer = AutoTokenizer.from_pretrained(MODEL_SAVE)
    model = RobertaForSequenceClassification.from_pretrained(MODEL_SAVE).to(device)
    model.eval()

    test_dataset = TestDataset(df_test['combined'].tolist(), tokenizer)
    collator = DataCollatorWithPadding(tokenizer)
    loader = DataLoader(test_dataset, batch_size=64, collate_fn=collator)

    all_preds = []
    with torch.no_grad():
        for i, batch in enumerate(loader):
            outputs = model(input_ids=batch['input_ids'].to(device), attention_mask=batch['attention_mask'].to(device))
            all_preds.extend(torch.argmax(outputs.logits, dim=-1).cpu().numpy())
            if (i+1) % 5 == 0: print(f'  Processed {(i+1)*64}/{len(test_dataset)} samples')

    label_map = {1: 'TRUE', 0: 'FALSE'}
    df_test['label'] = [label_map[p] for p in all_preds]
    submission = df_test[['label']]
    submission.to_csv('/kaggle/working/submission.csv', index=True, index_label='id')
    print(f'submission.csv saved: {len(submission):,} rows')
    print(f'Label distribution: {submission["label"].value_counts().to_dict()}')
    print('Download /kaggle/working/submission.csv and upload to competition leaderboard.')